In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import joblib

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Load your synthetic dataset
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/synthetic_battery_3S.csv")

# Show first rows
df.head()

,voltage,current,temperature,cycle,SOC,SOH
0,10.123620,-1.010873,39.599966,90,42.901189,97.029747
1,11.852143,-1.336703,28.690240,165,99.151237,94.248587
2,11.195982,-2.590769,31.932794,160,81.062875,94.583227
3,10.795975,0.858133,38.265613,5,60.373638,100.000000
4,9.468056,-0.187007,34.641787,140,17.997738,95.547941


In [6]:
# Input features (from sensors)
X = df[["voltage", "current", "temperature", "cycle"]]

# Targets (what we want to predict)
y = df[["SOC", "SOH"]]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

TRAINING


RANDOM FOREST

In [9]:
# Base Random Forest
rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    random_state=42
)

# Multi-output wrapper
model = MultiOutputRegressor(rf)

# Train
model.fit(X_train, y_train)

MultiOutputRegressor(estimator=RandomForestRegressor(max_depth=12,
                                                     n_estimators=150,
                                                     random_state=42))

In [11]:
# Predict on test data
y_pred = model.predict(X_test)

In [12]:
# Split predictions
soc_true = y_test["SOC"]
soh_true = y_test["SOH"]

soc_pred = y_pred[:, 0]
soh_pred = y_pred[:, 1]

# Metrics
soc_mae = mean_absolute_error(soc_true, soc_pred)
soh_mae = mean_absolute_error(soh_true, soh_pred)

soc_r2 = r2_score(soc_true, soc_pred)
soh_r2 = r2_score(soh_true, soh_pred)

print("SOC MAE:", round(soc_mae, 2))
print("SOC R2:", round(soc_r2, 3))
print()
print("SOH MAE:", round(soh_mae, 2))
print("SOH R2:", round(soh_r2, 3))

SOC MAE: 0.41
SOC R2: 1.0

SOH MAE: 0.13
SOH R2: 0.997


In [13]:
model_path = "/content/drive/MyDrive/battery_soc_soh_model.pkl"
joblib.dump(model, model_path)

['/content/drive/MyDrive/battery_soc_soh_model.pkl']

In [20]:
import joblib

# Load the model
model = joblib.load("/content/drive/MyDrive/battery_soc_soh_model.pkl")

# Check type
print(type(model))

# Check model estimator info
print(model.estimators_)  # This shows the list of Random Forest trees (optional)


<class 'sklearn.multioutput.MultiOutputRegressor'>
[RandomForestRegressor(max_depth=12, n_estimators=150, random_state=42), RandomForestRegressor(max_depth=12, n_estimators=150, random_state=42)]


In [21]:
import numpy as np

# Example input: voltage, current, temp, cycle
sample = np.array([[11.0, -2.0, 32.0, 150]])

pred = model.predict(sample)

print("Predicted SOC:", round(pred[0][0], 2))
print("Predicted SOH:", round(pred[0][1], 2))

Predicted SOC: 72.77
Predicted SOH: 95.09


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [22]:
import numpy as np

# Different example inputs: voltage, current, temperature, cycle
samples = np.array([
    [10.5, 1.5, 30.0, 50],    # partially charged, mild discharge
    [11.8, -3.0, 40.0, 200],  # nearly full charge, high charging current
    [9.5, 0.0, 25.0, 10],     # low voltage, resting battery
    [12.0, -1.0, 35.0, 290],  # full voltage, near end-of-life cycle
    [10.0, 2.5, 38.0, 120]    # mid-range voltage, high discharge
])

# Predict SOC & SOH
preds = model.predict(samples)

# Print results
for i, p in enumerate(preds):
    print(f"Sample {i+1} -> Predicted SOC: {round(p[0],2)}%, Predicted SOH: {round(p[1],2)}%")

Sample 1 -> Predicted SOC: 47.7%, Predicted SOH: 98.31%
Sample 2 -> Predicted SOC: 100.0%, Predicted SOH: 93.35%
Sample 3 -> Predicted SOC: 17.88%, Predicted SOH: 99.59%
Sample 4 -> Predicted SOC: 99.99%, Predicted SOH: 90.24%
Sample 5 -> Predicted SOC: 29.47%, Predicted SOH: 96.04%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
